# Visuelle Attribute (Encoding Channels) in Altair

Das Herzstück von Vega-Lite (in Python durch Altair verwendet) ist die Verwendung von [Kodierungen](https://altair-viz.github.io/user_guide/encoding.html), die Datenfelder (mit einem bestimmten Attributtyp) an verfügbare visuelle Attribute (`Encoding Channels`) eines gewählten visuellen Elements (`Mark`) binden.

In [1]:
import pandas as pd
import altair as alt

Als Beispieldatensatz wird der Gapminder Datensatz von Altair geladen:

In [2]:
from vega_datasets import data as vega_data
data = vega_data.gapminder()
data

,year,country,cluster,pop,life_expect,fertility
0,1955,Afghanistan,0,8891209,30.332,7.7000
1,1960,Afghanistan,0,9829450,31.997,7.7000
2,1965,Afghanistan,0,10997885,34.020,7.7000
3,1970,Afghanistan,0,12430623,36.088,7.7000
4,1975,Afghanistan,0,14132019,38.438,7.7000
...,...,...,...,...,...,...
688,1985,Venezuela,3,16997509,70.190,3.6485
689,1990,Venezuela,3,19325222,71.150,3.2500
690,1995,Venezuela,3,21555902,72.146,2.9415
691,2000,Venezuela,3,23542649,72.766,2.7230


Um die Zeilen in dem DataFrame einzugrenzen, filtern wir alle Werte auf das Jahr 2005:

In [3]:
data2005 = data[data['year'] == 2005]
data2005

,year,country,cluster,pop,life_expect,fertility
10,2005,Afghanistan,0,29928987,43.828,7.0685
21,2005,Argentina,3,39537943,75.320,2.2540
32,2005,Aruba,3,71566,74.239,2.0400
43,2005,Australia,4,20090437,81.235,1.7880
54,2005,Austria,1,8184691,79.829,1.4200
...,...,...,...,...,...,...
648,2005,Switzerland,1,7489370,81.701,1.4200
659,2005,Turkey,1,69660559,71.777,2.1430
670,2005,United Kingdom,1,60441457,79.425,1.8150
681,2005,United States,3,295734134,78.242,2.0540


In [4]:
data2005[data2005["cluster"] == 5]

,year,country,cluster,pop,life_expect,fertility
230,2005,Egypt,5,77505756,71.338,2.891
373,2005,Iran,5,68017860,70.964,2.040
384,2005,Iraq,5,26074906,59.545,4.264
406,2005,Israel,5,6276883,80.745,2.750
483,2005,Lebanon,5,3826018,71.993,2.209
615,2005,Saudi Arabia,5,26417599,72.777,3.352


### Attributtypen

Zunächst ist es wichtig, die Attributtypen des DataFrames zu bestimmen:

* `Nominal (N)`: enthalten Kategorien, die in keine Ordnung zu bringen sind (z.B. `country`)
* `Ordinal (O)`: enthalten Kateogorien, in einer bestimmten Ordnung (hier könnte man `year` als ordinalen Wert mit geordneten Jahresklassen verwenden)
* `Quantitativ (Q)`: numerische Werte, wie `life_expect`, `pop` und `fertility`

Die semantische Information, dass es sich um temporale Daten handelt, wird in Altair auch als Attributtyp angegeben: `Temporal (T)` und kann wie Nominal, Ordinal oder Quantitiav als Einheit für eine Achse gewählt werden.

## Magnituden-Kanäle
Visuelle Elemente (`Marks`) sind geometrische Grundelemente, die Elemente oder Verbindungen darstellen, und visuelle Attribute (`Encoding Channels`) steuern ihr Aussehen. Die Effektivität eines Kanals zur Kodierung von Daten hängt von seiner Art ab: Magnitudenkanäle, die Ordnungen oder Mengeninformationen wahrnehmbar vermitteln, eignen sich gut für geordnete Daten,und solche, die Identitätsinformationen vermitteln, für nominale Daten.

#### Position auf der gemeinsamen Skala
Die x-Achse legt die horizontale Position eines Marks fest. Darüber hinaus werden die Achsen und Titel automatisch ausgewählt. In der folgenden Visualisierung führt die Wahl eines quantitativen Datentyps zu einer kontinuierlichen linearen Achsenskala:

In [5]:
alt.Chart(data2005).mark_point().encode(
    alt.X('fertility:Q')
)

alt.Chart(...)

Der y-Wert legt die vertikale Position eines Marks fest. Hier wurde die Eigenschaft `cluster` unter Verwendung eines ordinalen (O) Attributtyps hinzugefügt. Das Ergebnis ist eine diskrete Achse, die für jeden eindeutigen Wert ein geordnetes Band mit einer Standardschrittweite enthält:

In [6]:
alt.Chart(data2005).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('cluster:N')
)

alt.Chart(...)

*Was passiert mit dem obigen Visualisierung, wenn Sie die Feldtypen O und Q vertauschen?

Wenn wir stattdessen das Feld `life_expect` als quantitative (Q) Variable hinzufügen, ist das Ergebnis ein Scatterplot mit linearen Skalen für beide Achsen:

In [7]:
alt.Chart(data2005).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q')
)

alt.Chart(...)

Standardmäßig enthalten Achsen für lineare quantitative Skalen den Wert Null, um eine angemessene Basislinie für den Vergleich verhältnismäßiger Daten zu gewährleisten. In einigen Fällen kann eine Null-Basislinie jedoch bedeutungslos sein oder es sollen nur bestimmte Intervalle verglichen werden. Um die automatische Einbeziehung von Null zu deaktivieren, kann die Skalenzuordnung mit `scale` konfiguriert werden (ACHTUNG: nicht bei Bar Charts!):

In [8]:
alt.Chart(data2005).mark_point().encode(
    alt.X('fertility:Q', scale=alt.Scale(zero=False)),
    alt.Y('life_expect:Q', scale=alt.Scale(zero=False))
)

alt.Chart(...)

Die Achsenskalen umfassen nun nicht mehr standardmäßig den Wert Null. Ein gewisses Maß an Auffüllung bleibt jedoch erhalten, da die Endpunkte der Achsen automatisch auf _nice_ Zahlen wie Vielfache von 5 oder 10 gefangen werden.

_Überprüfen Sie, was passiert, wenn Sie `nice=False` hinzufügen!_

In [9]:
alt.Chart(data2005).mark_point().encode(
    alt.X('fertility:Q', scale=alt.Scale(zero=False, nice=False)),
    alt.Y('life_expect:Q', scale=alt.Scale(zero=False, nice=False))
)

alt.Chart(...)

#### Position auf einer nicht ausgerichteten Skala

Mittels `Column` können die Werte einer Eigenschaften in mehrere quantitative Achsen aufgeteilt werden, hier auf Basis der Eigenschaft `cluster`:

In [10]:
alt.Chart(data2005).mark_point().encode(
    alt.X('life_expect:Q', scale=alt.Scale(zero=False)),
    alt.Column('cluster:O')
)

alt.Chart(...)

### Länge

In [11]:
alt.Chart(data2005).mark_bar().encode(
    alt.X('average(life_expect):Q'),
    alt.Y('cluster:O')
)

alt.Chart(...)

### Fläche

Der Kodierungskanal _Fläche_ (`size`) legt die Größe oder Ausdehnung einer Marke fest. Die Bedeutung des Kanals kann je nach Markentyp variieren. Bei Punktmarken entspricht die Größe der Pixelfläche des Zeichensymbols, so dass der Durchmesser des Punktes der Quadratwurzel des Größenwerts entspricht.

In [12]:
alt.Chart(data2005).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q')
)

alt.Chart(...)

In einigen Fällen kann es vorkommen, dass wir mit dem Standardgrößenbereich nicht zufrieden sind. Der Parameter `range` kann genutzt werde, um einen benutzerdefinierten Wertebereich festzulegen.

In [13]:
alt.Chart(data2005).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[10,1000]))
)

alt.Chart(...)

### Helligkeit / Sättigung

Der Kodierungskanal `color` legt die Farbe einer Markierung fest. Die Art der Farbkodierung hängt stark vom Datentyp ab: Ordinale und quantitative Daten verwenden wahrnehmungsmäßig geordnete Farbverläufe, während nominale Daten standardmäßig ein qualitatives Farbschema verwenden.

In [14]:
alt.Chart(data2005).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Color('pop:Q')
)

alt.Chart(...)

## Identitätskanäle

Der Identitätskanal wird für nominale Daten verwendet

### Räumlicher Bereich

In [15]:
europe = data2005[data2005['cluster'] == 1]
europe

,year,country,cluster,pop,life_expect,fertility
54,2005,Austria,1,8184691,79.829,1.4200
98,2005,Belgium,1,10364388,79.441,1.6460
186,2005,Croatia,1,4495904,75.748,1.3460
252,2005,Finland,1,5223442,79.313,1.8250
263,2005,France,1,60656178,80.657,1.8916
274,2005,Georgia,1,4677401,70.987,1.4070
285,2005,Germany,1,82431390,79.406,1.3600
296,2005,Greece,1,10668354,79.483,1.3250
340,2005,Iceland,1,296737,81.757,2.0520
395,2005,Ireland,1,4015676,78.885,1.9640


Mit `filled=True` können die Point Marks ausgefüllt werden:

In [16]:
alt.Chart(europe).mark_point(filled=True).encode(
    alt.X('country:N'),
    alt.Color('life_expect:Q')
)

alt.Chart(...)

In [17]:
alt.Chart(europe).mark_bar().encode(
    alt.X('country:N'),
    alt.Y('life_expect:Q')
)

alt.Chart(...)

### Farbe

Für nominale Daten wird der Farbwert genutzt (Color Hue). Hier wird das nominale Datenattribut `cluster` mithilfe des Farbtons kodiert.

In [18]:
alt.Chart(data2005).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N')
)

alt.Chart(...)

Standardmäßig verwendet Altair eine gewisse Transparenz, um Überlappungen entgegenzuwirken. Die Transparenz kann weiter angepasst werden.

Im folgenden Beispiel wird ein konstanter Wert an einen Kodierungskanal übergeben:

In [19]:
alt.Chart(data2005).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N'),
    alt.OpacityValue(0.5)
)

alt.Chart(...)

### Form

Der Kodierungkanal `shape` legt die geometrische Form fest, die von den visuelle Elementen verwendet wird. Im Gegensatz zu den anderen bisherigen Kanälen, kann der `shape` nicht von anderen Markierungstypen verwendet werden. Der Formkodierungskanal sollte nur mit nominalen Daten verwendet werden, da Wahrnehmungsrangfolgen und Größenvergleiche nicht unterstützt werden.

In [21]:
alt.Chart(data2005).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N'),
    alt.OpacityValue(0.5),
    alt.Shape('cluster:N')
)

alt.Chart(...)

##### Credits
The notebook in this collection is based on UW Interactive Data Lab Visualization Curriculum from Jeffrey Heer.